## 머신러닝

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import platform

df = pd.read_csv("data/2025_Airbnb_NYC_listings.csv") #----- 자기 경로 설정!!
df_cleaned = pd.read_csv("data/first_clean_data.csv")

In [ ]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

In [ ]:
df_cleaned = pd.read_csv("data/first_clean_data.csv")

## 머신러닝 전용 DF 전처리

In [ ]:
df_machine = df_cleaned.copy()
df_machine.head()

## 컬럼 드랍

In [ ]:
drop_cols = ['name', 'description', 'neighbourhood_cleansed', 'number_of_reviews',
            'estimated_revenue_l365d','calculated_host_listings_count_shared_rooms',
            'calculated_host_listings_count','calculated_host_listings_count_entire_homes','calculated_host_listings_count_private_rooms','host_id','host_since','id','latitude',
            'longitude','amenities','property_type','Unnamed: 0','number_of_reviews_ltm','availability_365','beds','bedrooms','accommodates','room_type','neighbourhood_group_cleansed',
            'host_is_superhost','host_acceptance_rate','estimated_occupancy_l365d','log_price']#'host_response_time','host_response_rate','reviews_per_month','price'
df_machine=df_machine.drop(columns = drop_cols)

In [ ]:
df_machine.info()

In [ ]:
df_machine = df_machine.dropna(subset = ['review_scores_rating'])

# 머신 러닝 시작 <br>
### Train/Test Split: train_test_split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

X = df_machine.drop(columns=["review_scores_rating"])
y = df_machine["review_scores_rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

결측치(Missing): 채우기 + 결측 자체도 피처로 <br>
결측 플래그 만들기

In [ ]:
from sklearn.preprocessing import RobustScaler

X_train['price'] = np.log1p(X_train['price'])
X_test['price'] = np.log1p(X_test['price'])

scale_cols = ['host_response_rate','reviews_per_month','price']
scale_cols = [c for c in scale_cols if c in X_train.columns]  # 안전장치

scaler = RobustScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

X_train[scale_cols].describe()

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. LightGBM 회귀 모델 생성 및 하이퍼파라미터 세팅
lgbm_model = LGBMRegressor(
    n_estimators=100,      # 나무의 개수
    learning_rate=0.1,     # 학습률 (오답 수정 강도)
    num_leaves=31,         # 하나의 나무가 가질 수 있는 최대 잎(노드)의 개수
    random_state=42,       # 결과 고정
    n_jobs=-1                        # 모든 CPU 코어 사용
)

In [ ]:
lgbm_model.fit(X_train, y_train)

In [ ]:
# 평가 데이터로 예측 수행
y_pred_lgbm = lgbm_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
import shap
# 모델이 학습한 데이터의 특성을 바탕으로 기여도를 계산.
explainer = shap.TreeExplainer(lgbm_model)

shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)

plt.title('SHAP Feature Importance', fontsize=15)
plt.tight_layout()
plt.show()

# (빨간색: 높은 값, 파란색: 낮은 값)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test)

In [ ]:
# 신규 호스트 가상 데이터 (딕셔너리)
my_listing = {
    'price': 146, 
    'review_scores_value': 4.8,
    'review_scores_cleanliness': 4.9,
    'review_scores_communication': 4.8,
    'review_scores_accuracy': 4.9,
    'review_scores_checkin': 4.8,
    'review_scores_location': 4.8,
    'host_response_rate': 95,
    'reivews_per_month' : 300,
    'host_response_time': 1
}

# 예측 함수
def predict_with_refined_model(input_data):
    test_df = pd.DataFrame([input_data])
    
    # 로그 변환
    if 'price' in test_df.columns:
        test_df['price'] = np.log1p(test_df['price'])
    
    # 원-핫 인코딩 및 컬럼 맞추기 : X_train 컬럼들을 0으로 초기화한 뒤 매핑
    final_input = pd.DataFrame(0, index=[0], columns=X_train.columns)
    
    for col, value in test_df.iloc[0].items():
        if col in final_input.columns:
            final_input[col] = value
        else:
            combined_col = f"{col}_{value}"
            if combined_col in final_input.columns:
                final_input[combined_col] = 1
    
    # final_scale_target 리스트를 재사용
    final_input[scale_cols] = scaler.transform(final_input[scale_cols])
    
    # 예측
    prediction = lgbm_model.predict(final_input)[0]
    return prediction

# 결과 출력
score = predict_with_refined_model(my_listing)
print(f"예상 평점: {score:.2f}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# 모든 피처에 대한 위젯 생성
style = {'description_width': '150px'} # 가독성을 위해 설명 너비 조절

# 수치형 변수 슬라이더
price_slider = widgets.IntSlider(value=150, min=10, max=500, step=5, description='가격 ($)', style=style)
rpm_slider = widgets.FloatSlider(value=2.0, min=0, max=50.0, step=0.1, description='월평균 리뷰수', style=style)
resp_rate_slider = widgets.IntSlider(value=100, min=0, max=100, step=1, description='호스트 응답률 (%)', style=style)

# 리뷰 세부 점수
val_slider = widgets.FloatSlider(value=4.8, min=0, max=5.0, step=0.1, description='가성비 점수', style=style)
cln_slider = widgets.FloatSlider(value=4.8, min=0, max=5.0, step=0.1, description='청결도 점수', style=style)
com_slider = widgets.FloatSlider(value=4.8, min=0, max=5.0, step=0.1, description='소통 점수', style=style)
acc_slider = widgets.FloatSlider(value=4.8, min=0, max=5.0, step=0.1, description='정확도 점수', style=style)
chk_slider = widgets.FloatSlider(value=4.8, min=0, max=5.0, step=0.1, description='체크인 점수', style=style)
loc_slider = widgets.FloatSlider(value=4.8, min=0, max=5.0, step=0.1, description='위치 점수', style=style)

# 실시간 업데이트 함수
output = widgets.Output()

def update_prediction(change):
    with output:
        clear_output(wait=True)
        
        current_listing = {
            'price': price_slider.value,
            'reviews_per_month': rpm_slider.value,
            'host_response_rate': resp_rate_slider.value,
            'review_scores_value': val_slider.value,
            'review_scores_cleanliness': cln_slider.value,
            'review_scores_communication': com_slider.value,
            'review_scores_accuracy': acc_slider.value,
            'review_scores_checkin': chk_slider.value,
            'review_scores_location': loc_slider.value,
            'host_response_time': 1 
        }
        
        # 예측 수행
        try:
            pred_score = predict_with_refined_model(current_listing)
            
            # 시각적 피드백
            print(f"✅ 실시간 예상 평점: {pred_score:.3f} / 5.0")
            
            # 점수 바 차트 시각화
            fig, ax = plt.subplots(figsize=(7, 1.5))
            color = '#2ECC71' if pred_score >= 4.7 else '#F1C40F' if pred_score >= 4.4 else '#E74C3C'
            ax.barh(['Rating'], [pred_score], color=color)
            ax.set_xlim(0, 5)
            ax.axvline(x=4.5, color='black', linestyle='--', alpha=0.3) # 기준선
            plt.title("Current Listing Performance Estimate")
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"❌ 오류 발생: {e}")
            print("Tip: predict_with_refined_model 함수 내부의 컬럼명과 딕셔너리 키값이 일치하는지 확인하세요.")

all_widgets = [price_slider, rpm_slider, resp_rate_slider, val_slider, 
               cln_slider, com_slider, acc_slider, chk_slider, loc_slider]

for w in all_widgets:
    w.observe(update_prediction, names='value')

input_ui = widgets.VBox([
    widgets.HTML("<b style='font-size:15px;'>💰 기본 정보</b>"),
    price_slider, rpm_slider, resp_rate_slider,
    widgets.HTML("<br><b style='font-size:15px;'>⭐ 상세 만족도 지표</b>"),
    val_slider, cln_slider, com_slider, acc_slider, chk_slider, loc_slider
])

display(widgets.HBox([input_ui, output], layout=widgets.Layout(gap='50px')))

# 초기 실행
update_prediction(None)